# Checkpoint 3 - Bronze ingestion and silver transformation

This notebook implements the first build increment for the Online Retail II pipeline. It loads the raw CSV into a bronze Delta table, creates a typed and deduplicated silver table, preserves rejected rows, and writes a quality log. Run on a small sample before the full dataset.

In [ ]:
from pyspark.sql import functions as F, types as T

# Replace these values with the paths and names in your Databricks workspace.
source_path = "/Volumes/workspace/default/team9_volume/online_retail_II.csv"
catalog_name = "workspace"
schema_name = "default"
table_prefix = "retail_project"
sample_fraction = 1.0  # Use 0.01 while developing; use 1.0 for the final run.

bronze_table = f"{catalog_name}.{schema_name}.{table_prefix}_bronze"
silver_table = f"{catalog_name}.{schema_name}.{table_prefix}_silver"
exceptions_table = f"{catalog_name}.{schema_name}.{table_prefix}_silver_exceptions"
quality_table = f"{catalog_name}.{schema_name}.{table_prefix}_quality_log"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

## 1. Bronze: load the raw file as-is

In [ ]:
raw = (spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("multiLine", True)
    .csv(source_path))
raw = raw.withColumnRenamed("Customer ID", "Customer_ID")

if sample_fraction < 1.0:
    raw = raw.sample(withReplacement=False, fraction=sample_fraction, seed=42)

bronze = (raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(source_path)))
bronze.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(bronze_table)
bronze_count = bronze.count()
print(f"Bronze rows: {bronze_count}")
display(bronze.limit(10))

## 2. Silver: standardise, type, deduplicate, and classify records

In [ ]:
silver_base = (bronze
    .withColumnRenamed("Invoice", "invoice_id")
    .withColumnRenamed("StockCode", "stock_code")
    .withColumnRenamed("Description", "description")
    .withColumnRenamed("Quantity", "quantity")
    .withColumnRenamed("InvoiceDate", "invoice_date")
    .withColumnRenamed("Price", "unit_price")
    .withColumnRenamed("Customer_ID", "customer_id")
    .withColumnRenamed("Country", "country")
    .withColumn("invoice_id", F.trim(F.col("invoice_id")))
    .withColumn("stock_code", F.trim(F.col("stock_code")))
    .withColumn("description", F.trim(F.col("description")))
    .withColumn("country", F.trim(F.col("country")))
    .withColumn("quantity", F.col("quantity").cast(T.IntegerType()))
    .withColumn("unit_price", F.col("unit_price").cast(T.DecimalType(18, 4)))
    .withColumn("customer_id", F.col("customer_id").cast(T.StringType()))
    .withColumn("invoice_date", F.to_timestamp("invoice_date")))

deduped = silver_base.dropDuplicates(["invoice_id", "stock_code", "quantity", "invoice_date", "unit_price", "customer_id"])
silver = (deduped
    .withColumn("is_cancelled", F.upper(F.col("invoice_id")).startswith("C"))
    .withColumn("line_value", F.col("quantity") * F.col("unit_price"))
    .withColumn("has_exception",
        F.col("invoice_id").isNull() | F.col("stock_code").isNull() |
        F.col("invoice_date").isNull() | F.col("quantity").isNull() |
        F.col("unit_price").isNull() | (F.col("quantity") <= 0) |
        (F.col("unit_price") <= 0)))

exceptions = silver.filter(F.col("has_exception"))
clean_silver = silver.filter(~F.col("has_exception"))

clean_silver.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(silver_table)
exceptions.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(exceptions_table)
print(f"Silver rows: {clean_silver.count()} | Exceptions: {exceptions.count()}")
display(clean_silver.limit(10))

## 3. Repeatable quality checks

In [ ]:
required_columns = {"invoice_id", "stock_code", "quantity", "invoice_date", "unit_price", "customer_id", "country"}
actual_columns = set(clean_silver.columns)
dedupe_count = silver_base.count() - deduped.count()
checks = [
    ("required_columns_present", required_columns.issubset(actual_columns), f"missing={sorted(required_columns - actual_columns)}"),
    ("silver_has_no_duplicate_business_rows", clean_silver.dropDuplicates(["invoice_id", "stock_code", "quantity", "invoice_date", "unit_price", "customer_id"]).count() == clean_silver.count(), ""),
    ("silver_has_positive_quantity", clean_silver.filter(F.col("quantity") <= 0).count() == 0, ""),
    ("silver_has_positive_unit_price", clean_silver.filter(F.col("unit_price") <= 0).count() == 0, ""),
    ("row_reconciliation", bronze_count == clean_silver.count() + exceptions.count() + dedupe_count, f"bronze={bronze_count}, silver={clean_silver.count()}, exceptions={exceptions.count()}, duplicates_removed={dedupe_count}"),
]
quality_log = spark.createDataFrame([(name, bool(passed), detail) for name, passed, detail in checks], ["check_name", "passed", "details"]).withColumn("checked_at", F.current_timestamp())
quality_log.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable(quality_table)
display(quality_log)

failed = [name for name, passed, _ in checks if not passed]
if failed:
    raise ValueError(f"Quality checks failed: {failed}")
print("All Checkpoint 3 quality checks passed.")

## Notes for submission

- The notebook uses the Team 9 Databricks Volume and workspace.default schema configured above.
- Run the notebook in Databricks before submission and retain the displayed quality-log output.
- Record any failed checks or dataset-specific parsing changes honestly in the progress note.